# 03 - Filter Validation and Quality Check

**Purpose:** Verify that our Amendment 3 filters are working correctly

---

## What This Notebook Does

In the previous notebook, we created filters to make pre-June 16 data comparable to post-June 16 data. Now we need to check:

1. **Distribution Comparison:** Does our filtered data look similar to the actual post-June 16 data?
2. **Edge Case Analysis:** Are the crashes we're excluding truly minor?
3. **Confidence Assessment:** How certain can we be that our filter is accurate?
4. **Bias Check:** Does our filter favor or disfavor Waymo?

---

## Step 0: Import Libraries

In [ ]:
import pandas as pd
import os

## Step 1: Define File Paths

In [ ]:
# =============================================================================
# FILE PATHS - Should match outputs from previous notebooks
# =============================================================================

# Filtered data from notebook 02
PRIOR_WITH_FLAGS_PATH = "../data/waymo_crashes_prior_with_filter_flags_20250129.csv"

# Actual post-June 16 data (for comparison)
POST_DATA_PATH = "../data/waymo_crashes_post_merged_20250129.csv"

# Verify files exist
print("Checking input files...")
for name, path in [("Prior with flags", PRIOR_WITH_FLAGS_PATH), 
                   ("Post data", POST_DATA_PATH)]:
    if os.path.exists(path):
        print(f"[OK] {name}")
    else:
        print(f"[MISSING] {name}: {path}")

## Step 2: Load the Data

In [ ]:
# =============================================================================
# Load the datasets
# =============================================================================

# Prior data with filter flags
prior_data = pd.read_csv(PRIOR_WITH_FLAGS_PATH, dtype={'hub_zip_code': str})
print(f"Prior data (with flags): {len(prior_data):,} crashes")

# Actual post-June 16 data
post_data = pd.read_csv(POST_DATA_PATH, dtype={'hub_zip_code': str})
print(f"Post data (actual):      {len(post_data):,} crashes")

In [ ]:
# =============================================================================
# Create subsets for analysis
# =============================================================================

# V2 included/excluded
v2_included = prior_data[prior_data['reportable_v2']]
v2_excluded = prior_data[~prior_data['reportable_v2']]

# V3 included/excluded
v3_included = prior_data[prior_data['reportable_v3']]
v3_excluded = prior_data[~prior_data['reportable_v3']]

print(f"V2 included: {len(v2_included):,}  |  V2 excluded: {len(v2_excluded):,}")
print(f"V3 included: {len(v3_included):,}  |  V3 excluded: {len(v3_excluded):,}")
print(f"Post actual: {len(post_data):,}")

## Step 3: Compare Distributions

If our filter is working well, the V3 filtered data should have similar patterns to the actual post-June 16 data.

**Key metrics to compare:**
- **Stationary rate:** What % of crashes had Waymo stopped/parked?
- **Injury rate:** What % of crashes involved injuries?
- **Tow rate:** What % of crashes required a tow?

In [ ]:
# =============================================================================
# Calculate comparison metrics
# =============================================================================

def calc_stationary_rate(df):
    """What % of crashes had Waymo stopped or parked?"""
    stationary = df['SV Pre-Crash Movement'].isin(['Stopped', 'Parked'])
    return stationary.mean()

def calc_injury_rate_prior(df):
    """Injury rate for pre-June 16 data format"""
    injury_values = ['Minor', 'Moderate', 'Serious', 'Fatality']
    return df['Highest Injury Severity Alleged'].isin(injury_values).mean()

def calc_injury_rate_post(df):
    """Injury rate for post-June 16 data format"""
    no_injury = ['Property Damage. No Injured Reported', 'No Injured Reported']
    return (~df['Highest Injury Severity Alleged'].isin(no_injury)).mean()

def calc_tow_rate_prior(df):
    """Tow rate for pre-June 16 data"""
    sv_tow = df['SV Was Vehicle Towed?'] == 'Yes'
    cp_tow = df['CP Was Vehicle Towed?'] == 'Yes'
    return (sv_tow | cp_tow).mean()

def calc_tow_rate_post(df):
    """Tow rate for post-June 16 data"""
    if 'Was Any Vehicle Towed?' in df.columns:
        return df['Was Any Vehicle Towed?'].str.contains('Yes', na=False).mean()
    else:
        return calc_tow_rate_prior(df)

# Calculate all metrics
v2_stationary = calc_stationary_rate(v2_included)
v3_stationary = calc_stationary_rate(v3_included)
post_stationary = calc_stationary_rate(post_data)

v2_injury = calc_injury_rate_prior(v2_included)
v3_injury = calc_injury_rate_prior(v3_included)
post_injury = calc_injury_rate_post(post_data)

v2_tow = calc_tow_rate_prior(v2_included)
v3_tow = calc_tow_rate_prior(v3_included)
post_tow = calc_tow_rate_post(post_data)

print("Metrics calculated successfully.")

In [ ]:
# =============================================================================
# Display comparison table
# =============================================================================

print("DISTRIBUTION COMPARISON")
print("="*75)
print()
print("How similar is our filtered data to the actual post-June 16 data?")
print()
print(f"{'Metric':<20} {'V2 Filtered':<15} {'V3 Filtered':<15} {'Actual Post':<15} {'V3 vs Post':<10}")
print("-"*75)
print(f"{'Crash count':<20} {len(v2_included):<15,} {len(v3_included):<15,} {len(post_data):<15,} {'(diff periods)':<10}")
print(f"{'Stationary rate':<20} {v2_stationary:<15.1%} {v3_stationary:<15.1%} {post_stationary:<15.1%} {v3_stationary-post_stationary:>+10.1%}")
print(f"{'Injury rate':<20} {v2_injury:<15.1%} {v3_injury:<15.1%} {post_injury:<15.1%} {v3_injury-post_injury:>+10.1%}")
print(f"{'Tow rate':<20} {v2_tow:<15.1%} {v3_tow:<15.1%} {post_tow:<15.1%} {v3_tow-post_tow:>+10.1%}")
print()
print("INTERPRETATION:")
print("  - Smaller differences = better match to actual Amendment 3 data")
print("  - V3 should match better than V2 (that's why it's recommended)")
print("  - Some difference is expected (different time periods, operational changes)")

## Step 4: Analyze Excluded Crashes

Let's verify that the crashes we're excluding are truly minor, other-party-at-fault incidents that wouldn't be reported under the new rules.

In [ ]:
# =============================================================================
# V3 Excluded Crashes - Overview
# =============================================================================

print("V3 EXCLUDED CRASHES - Are these truly minor?")
print("="*60)
print(f"Total excluded: {len(v3_excluded):,}")
print()

# These should ALL be zero (if not, something is wrong)
print("SANITY CHECKS (all should be 0):")
print(f"  With injuries:  {v3_excluded['filter_has_injury'].sum()}")
print(f"  With tow:       {v3_excluded['filter_has_tow'].sum()}")
print(f"  With airbag:    {v3_excluded['filter_has_airbag'].sum()}")
print(f"  VRU involved:   {v3_excluded['filter_is_vru'].sum()}")
print()

# Check characteristics
stationary_pct = v3_excluded['filter_waymo_was_stationary'].mean()
low_deltav = (v3_excluded['hub_delta_v_less_than_1mph'] == True).sum()
unknown_deltav = v3_excluded['hub_delta_v_less_than_1mph'].isna().sum()

print("CHARACTERISTICS OF EXCLUDED CRASHES:")
print(f"  Waymo was stationary:    {stationary_pct:.1%}")
print(f"  Delta-V < 1 mph:         {low_deltav:,} ({100*low_deltav/len(v3_excluded):.1f}%)")
print(f"  Delta-V unknown:         {unknown_deltav:,} ({100*unknown_deltav/len(v3_excluded):.1f}%)")
print()
print("CONCLUSION:")
print("  If most excluded crashes have Waymo stationary and low delta-V,")
print("  they are truly minor, other-party-at-fault incidents.")

In [ ]:
# =============================================================================
# Edge Case: Excluded crashes where Waymo was MOVING
# =============================================================================

# These are the trickiest cases - why are we excluding crashes where Waymo was moving?

excluded_moving = v3_excluded[v3_excluded['filter_waymo_was_moving']]

print("EDGE CASE: Excluded crashes where Waymo was moving")
print("="*60)
print(f"Count: {len(excluded_moving):,} ({100*len(excluded_moving)/len(v3_excluded):.1f}% of excluded)")
print()

if len(excluded_moving) > 0:
    # Check if any had Waymo front contact (which would suggest Waymo struck someone)
    front_contact = (
        (excluded_moving['SV Contact Area - Front'] == 'Y') |
        (excluded_moving['SV Contact Area - Front Left'] == 'Y') |
        (excluded_moving['SV Contact Area - Front Right'] == 'Y')
    ).sum()
    
    rear_contact = (
        (excluded_moving['SV Contact Area - Rear'] == 'Y') |
        (excluded_moving['SV Contact Area - Rear Left'] == 'Y') |
        (excluded_moving['SV Contact Area - Rear Right'] == 'Y')
    ).sum()
    
    print("Where was Waymo hit?")
    print(f"  Front contact:  {front_contact}")
    print(f"  Rear contact:   {rear_contact} (hit from behind)")
    print()
    
    if front_contact == 0:
        print("KEY FINDING: Zero front contact!")
        print("  This means these are cases where other drivers hit a moving Waymo")
        print("  from behind or the side - Waymo didn't strike anyone.")
        print("  These are correctly excluded as other-party-at-fault.")
    else:
        print(f"NOTE: {front_contact} crashes have front contact - review these manually.")

## Step 5: Confidence Assessment

How confident can we be in our filter? We estimate precision and recall.

**Important caveat:** These are ESTIMATES based on professional judgment, not ground truth calculations. We don't have definitive labels for which crashes would actually be reported under the new rules.

**Definitions:**
- **Precision:** Of crashes we INCLUDE, what % are truly reportable?
- **Recall:** Of crashes that SHOULD be reported, what % did we include?

In [ ]:
# =============================================================================
# Confidence levels by filter component
# =============================================================================

print("CONFIDENCE LEVELS BY FILTER COMPONENT")
print("="*70)
print()
print("Each part of our filter has different certainty:")
print()
print(f"{'Component':<35} {'Confidence':<12} {'Reason'}")
print("-"*70)
print(f"{'5-day triggers (injury/tow/etc)':<35} {'100%':<12} {'Exact match to official rules'}")
print(f"{'Single-vehicle crashes':<35} {'~95%':<12} {'Clear categories, minor edge cases'}")
print(f"{'Waymo struck another':<35} {'~90%':<12} {'Contact area is good but not perfect'}")
print(f"{'Delta-V >= 1 mph proxy':<35} {'~80%':<12} {'Proxy for $1,000 damage threshold'}")
print()
print("NOTE: The confidence percentages are professional estimates, not calculations.")

In [ ]:
# =============================================================================
# Estimate Precision
# =============================================================================

# Count V3 included by reason
n_5day = v3_included['filter_is_5day_trigger'].sum()
n_single = ((~v3_included['filter_is_5day_trigger']) & 
            v3_included['filter_is_single_vehicle']).sum()
n_struck = ((~v3_included['filter_is_5day_trigger']) & 
            (~v3_included['filter_is_single_vehicle']) & 
            v3_included['filter_waymo_struck_another']).sum()
n_deltav = ((~v3_included['filter_is_5day_trigger']) & 
            (~v3_included['filter_is_single_vehicle']) & 
            (~v3_included['filter_waymo_struck_another']) & 
            v3_included['filter_deltav_over_1mph']).sum()

# Apply confidence levels
conf_5day = 1.00
conf_single = 0.95
conf_struck = 0.90
conf_deltav = 0.80

est_true_positives = (n_5day * conf_5day + 
                      n_single * conf_single + 
                      n_struck * conf_struck + 
                      n_deltav * conf_deltav)

precision = est_true_positives / len(v3_included)

print("PRECISION ESTIMATE")
print("="*60)
print("Of crashes we INCLUDE, what % are truly reportable?")
print()
print(f"{'Reason':<30} {'Count':<10} {'Conf':<10} {'Est. True Pos':<15}")
print("-"*60)
print(f"{'5-day triggers':<30} {n_5day:<10,} {conf_5day:<10.0%} {n_5day * conf_5day:<15,.1f}")
print(f"{'Single-vehicle':<30} {n_single:<10,} {conf_single:<10.0%} {n_single * conf_single:<15,.1f}")
print(f"{'Waymo struck':<30} {n_struck:<10,} {conf_struck:<10.0%} {n_struck * conf_struck:<15,.1f}")
print(f"{'Delta-V >= 1 mph':<30} {n_deltav:<10,} {conf_deltav:<10.0%} {n_deltav * conf_deltav:<15,.1f}")
print("-"*60)
print(f"{'TOTAL':<30} {len(v3_included):<10,} {'':<10} {est_true_positives:<15,.1f}")
print()
print(f"ESTIMATED PRECISION: {precision:.1%}")

In [ ]:
# =============================================================================
# Estimate Recall
# =============================================================================

# Estimate false negatives from excluded crashes
low_deltav_excluded = (v3_excluded['hub_delta_v_less_than_1mph'] == True).sum()
unknown_deltav_excluded = v3_excluded['hub_delta_v_less_than_1mph'].isna().sum()

# Error rates (professional judgment)
fn_rate_low_deltav = 0.05  # 5% of <1mph might actually be >$1000 damage
fn_rate_unknown = 0.15     # 15% of unknown might be >=1mph

est_false_negatives = (low_deltav_excluded * fn_rate_low_deltav + 
                       unknown_deltav_excluded * fn_rate_unknown)

recall = est_true_positives / (est_true_positives + est_false_negatives)

print("RECALL ESTIMATE")
print("="*60)
print("Of crashes that SHOULD be reported, what % did we include?")
print()
print(f"Excluded crashes with delta-V < 1 mph:  {low_deltav_excluded:,}")
print(f"  Estimated false negative rate: 5%")
print(f"  Estimated false negatives: {low_deltav_excluded * fn_rate_low_deltav:.1f}")
print()
print(f"Excluded crashes with unknown delta-V:  {unknown_deltav_excluded:,}")
print(f"  Estimated false negative rate: 15%")
print(f"  Estimated false negatives: {unknown_deltav_excluded * fn_rate_unknown:.1f}")
print()
print(f"Total estimated false negatives: {est_false_negatives:.1f}")
print()
print(f"ESTIMATED RECALL: {recall:.1%}")

In [ ]:
# =============================================================================
# Summary: Precision, Recall, F1
# =============================================================================

f1 = 2 * (precision * recall) / (precision + recall)

print("FILTER PERFORMANCE SUMMARY")
print("="*60)
print()
print(f"  Estimated Precision:  {precision:.1%}")
print(f"  Estimated Recall:     {recall:.1%}")
print(f"  Estimated F1 Score:   {f1:.1%}")
print()
print("INTERPRETATION:")
print(f"  - Precision {precision:.0%}: Of crashes we INCLUDED, ~{precision:.0%} are truly reportable")
print(f"  - Recall {recall:.0%}: Of crashes that SHOULD be reported, we captured ~{recall:.0%}")
print(f"  - F1 {f1:.0%}: Overall balanced performance")
print()
print("IMPORTANT CAVEAT:")
print("  These are ESTIMATES based on professional judgment about confidence")
print("  levels, not calculations from ground truth data.")

## Step 6: Bias Assessment

Does our filter systematically favor or disfavor Waymo? This is important for journalistic integrity.

In [ ]:
# =============================================================================
# Bias Assessment
# =============================================================================

print("BIAS ASSESSMENT")
print("="*70)
print()
print("Does our filter favor or disfavor Waymo?")
print()
print("POTENTIAL ERROR SOURCES:")
print()
print("1. Delta-V >= 1 mph proxy:")
print("   - If we include crashes with >= 1 mph that had < $1,000 damage:")
print("     Effect: MORE crashes counted = UNFAVORABLE to Waymo")
print("   - If we exclude crashes with < 1 mph that had > $1,000 damage:")
print("     Effect: FEWER crashes counted = FAVORABLE to Waymo")
print()
print("2. Contact area analysis:")
print("   - May miss some edge cases where Waymo was at fault")
print("     Effect: Slightly FAVORABLE to Waymo")
print()
print("-"*70)
print()
print("OVERALL ASSESSMENT: APPROXIMATELY NEUTRAL")
print()
print("The delta-V >= 1 mph threshold likely INCLUDES more crashes than a")
print("true $1,000 damage threshold would. This is because:")
print("  - $1,000 is a relatively low repair threshold")
print("  - Most crashes with delta-V >= 1 mph probably exceed $1,000 in damage")
print()
print("If anything, our filter is SLIGHTLY CONSERVATIVE (unfavorable to Waymo),")
print("which makes the comparison more defensible for journalistic purposes.")

## Step 7: Final Summary

In [ ]:
# =============================================================================
# Final Summary
# =============================================================================

print("="*70)
print("FILTER VALIDATION COMPLETE")
print("="*70)
print()
print("KEY FINDINGS:")
print()
print("1. DISTRIBUTION SIMILARITY")
print(f"   V3 stationary rate: {v3_stationary:.1%} vs Actual: {post_stationary:.1%}")
print(f"   V3 injury rate:     {v3_injury:.1%} vs Actual: {post_injury:.1%}")
print(f"   Conclusion: V3 reasonably matches actual Amendment 3 data patterns")
print()
print("2. EXCLUDED CRASHES QUALITY")
print(f"   {100*v3_excluded['filter_waymo_was_stationary'].mean():.0f}% had Waymo stationary")
print(f"   {100*(v3_excluded['hub_delta_v_less_than_1mph']==True).sum()/len(v3_excluded):.0f}% had delta-V < 1 mph")
print(f"   Conclusion: Excluded crashes are truly minor, other-party-at-fault")
print()
print("3. ESTIMATED PERFORMANCE")
print(f"   Precision: {precision:.1%}")
print(f"   Recall:    {recall:.1%}")
print(f"   F1 Score:  {f1:.1%}")
print()
print("4. BIAS DIRECTION")
print(f"   Approximately neutral, possibly slightly conservative")
print()
print("-"*70)
print()
print("RECOMMENDATION:")
print("  Use V3 filter for apples-to-apples comparisons across the")
print("  June 16, 2025 reporting rule change.")
print()
print("  The filter achieves ~{:.0f}% estimated precision and ~{:.0f}% recall,".format(100*precision, 100*recall))
print("  which is sufficient for valid journalistic analysis.")